# Module SC1 — Data Prep & EDA (10-class)

> **Run on Kaggle** — Settings: Accelerator **GPU T4 x2**, Internet **ON**, Environment **Always use latest**. Attach inputs listed below, then **Save Version (Save & Run All)** so the outputs can be chained into the next module.

**Inputs:** `thesis-common-10c` (thesis_common.py) + the image dataset (`waste-10class`).

**Outputs:** `config.json`, `policy.json`, `data_summary.json` — chain these into Modules SC2–SC8.

In [ ]:

import sys, glob, os, json
import numpy as np
# Provenance bootstrap: explicit content identity; no timestamp/size selection.
import hashlib
from pathlib import Path
_PINNED_HELPER_SHA256 = "57b5c16cc9a08c7a3b2afa02c90ece96859b207d4533c78657f862752b68c34b"
ARTIFACT_OVERRIDES = {}  # T.find_one-only paths; generic cached artifact conflicts still require detaching.
_local_input_root = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
_input_roots = [Path('/kaggle/input')] if Path('/kaggle/input').is_dir() else [_local_input_root]
def _content_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def _input_candidates(pattern):
    hits = set()
    for root in _input_roots:
        for path in root.rglob(pattern):
            if path.is_file() and not {'Archives', '.git', 'node_modules'}.intersection(path.parts):
                hits.add(str(path.resolve()))
    return sorted(hits)
_helpers = _input_candidates('thesis_common.py')
_matching_helpers = [p for p in _helpers if _content_sha256(p) == _PINNED_HELPER_SHA256]
assert _matching_helpers, ('Attach the thesis_common.py supplied in Notebooks. '
                           'No input helper matches SHA256 ' + _PINNED_HELPER_SHA256)
_helper_path = _matching_helpers[0]
sys.path.insert(0, str(Path(_helper_path).parent))
import importlib, thesis_common as T
importlib.reload(T)
assert _content_sha256(T.__file__) == _PINNED_HELPER_SHA256, 'Wrong thesis_common imported.'
def _strict_find_one(filename_glob, required=True):
    if filename_glob in ARTIFACT_OVERRIDES:
        path = Path(ARTIFACT_OVERRIDES[filename_glob])
        assert path.is_file(), 'ARTIFACT_OVERRIDES path missing: ' + str(path)
        return str(path)
    hits = _input_candidates(filename_glob)
    if not hits:
        if required:
            raise FileNotFoundError('Attach the exact producer for ' + filename_glob + '; see the notebook preamble.')
        return None
    if len(hits) > 1:
        digests = {_content_sha256(p) for p in hits}
        if len(digests) != 1:
            raise RuntimeError('Different artifacts share ' + filename_glob + ':\n' + '\n'.join(hits)
                               + '\nDetach stale producers or set ARTIFACT_OVERRIDES explicitly.')
    return hits[0]
T.find_one = _strict_find_one
# Original local find()/find_all() consumers cannot obey T.find_one overrides.
# Reject conflicting standard cached artifacts up front instead of letting those
# consumers silently choose a same-name SC7/SC26 or stale output.
_generic_patterns = ('logits_*.npz', 'metrics_*.json', 'history_*.csv', 'calibration.json', 'config.json')
_by_basename = {}
for _pattern in _generic_patterns:
    for _p in _input_candidates(_pattern):
        _by_basename.setdefault(Path(_p).name, []).append(_p)
for _name, _paths in _by_basename.items():
    if len(_paths) > 1 and len({_content_sha256(p) for p in _paths}) > 1:
        raise RuntimeError('Conflicting input producers for ' + _name + ':\n' + '\n'.join(_paths)
                           + '\nAttach one producer/version per basename for this run.')
print('Pinned helper:', _helper_path, '| SHA256:', _PINNED_HELPER_SHA256)
_cfg = T.find_one('config.json', required=False)
if _cfg: T.load_config(_cfg); print('loaded config:', _cfg)
print('device:', T.DEVICE, '| materials:', len(T.MATERIALS), '| epochs:', T.EPOCHS)


In [ ]:

ROOT = T.find_image_root(); print('dataset root:', ROOT)
df_tr, df_va, df_te = T.load_manifests(ROOT)
T.check_leakage(df_tr, df_va, df_te)
import pandas as pd
def item_df(df): return df.drop_duplicates('item_id')
itr, iva, ite = item_df(df_tr), item_df(df_va), item_df(df_te)
print('\nITEMS per split:', len(itr), len(iva), len(ite),
      '| IMAGES per split:', len(df_tr), len(df_va), len(df_te))

### Class balance (item-level) — spot rare classes that will hurt macro-F1

In [ ]:

import pandas as pd
alli = pd.concat([itr.assign(split='train'), iva.assign(split='val'), ite.assign(split='test')])
print('--- material x split (items) ---')
print(pd.crosstab(alli.material, alli.split))
for col in ['route','condition','threat']:
    print(f'\n--- {col} (item-level, all) ---'); print(alli[col].value_counts().sort_index())

### Label co-occurrence — does material predict route/threat? (justifies the policy)

In [ ]:

print('--- material x route (items) ---'); print(pd.crosstab(alli.material, alli.route))
print('\n--- material x threat (items) ---'); print(pd.crosstab(alli.material, alli.threat))
print('\n--- condition x route (items) ---'); print(pd.crosstab(alli.condition, alli.route))

### Bounding-box size sanity (detector + crop quality)

In [ ]:

import numpy as np
area = (pd.to_numeric(df_tr.bbox_w, errors='coerce') * pd.to_numeric(df_tr.bbox_h, errors='coerce')).dropna()
print('bbox area fraction  min/median/max: %.3f / %.3f / %.3f' % (area.min(), area.median(), area.max()))
print('tiny boxes (<2%%): %d | huge boxes (>92%%): %d | of %d' %
      ((area<0.02).sum(), (area>0.92).sum(), len(area)))

### Save config + policy + data summary (these chain downstream)

In [ ]:

T.dump_config('config.json'); json.dump(T.POLICY, open('policy.json','w'), indent=2)
summary = {'items':{'train':len(itr),'val':len(iva),'test':len(ite)},
           'images':{'train':len(df_tr),'val':len(df_va),'test':len(df_te)},
           'materials':T.MATERIALS, 'routes':T.ROUTES,
           'material_counts': alli.material.value_counts().to_dict(),
           'route_counts': alli.route.value_counts().to_dict(),
           'threat_counts': {int(k):int(v) for k,v in alli.threat.value_counts().items()}}
json.dump(summary, open('data_summary.json','w'), indent=2)
print('WROTE config.json, policy.json, data_summary.json'); print(json.dumps(summary['items'], indent=2))
print('\nTRAINING RECIPE (v3):', {k: getattr(T,k) for k in
      ['EPOCHS','STRONG_AUG','USE_EMA','USE_UNCERTAINTY_W','USE_CB_FOCAL','USE_TTA','LABEL_SMOOTH','WEIGHT_DECAY']})